## Ingest circuits.csv file
1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table    

In [0]:
%run ../00-common/01.environment_config

In [0]:
source_file = f'{landing_folder_path}/circuits.csv'
table_name = f'{catalog_name}.{bronze_schema}.circuits'
print(source_file)
print(table_name)

#### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

In [0]:
circuits_df = (
    spark.read
        .format('csv')
        .option('header', 'true')
        # .option('inferSchema', 'true')
        .schema(circuits_schema)
        .load(source_file)
    )

In [0]:
circuits_df.printSchema()
display(circuits_df)

#### Step 2 - Add Metadata Columns 
- Source File
- Ingestion Timestamp

In [0]:
import pyspark.sql.functions as F
circuits_final_df = (
    circuits_df
    .withColumn('ingestion_timestamp', F.current_timestamp())
    .withColumn('source_file', F.col('_metadata.file_path'))
)

In [0]:
display(circuits_final_df)

####Step 3 - Write to bronze delta table  

In [0]:
(
    circuits_final_df
    .write
    .mode('overwrite')
    .format('delta')
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))